## Split PDF into images

In [ ]:
import tqdm 
from pdf2image import convert_from_path

progress = tqdm.tqdm(range(60,774,10))
for i in progress: 
    last_page = i + 10
    progress.set_description(str(i))
    if i == 770: 
        last_page = 774 
    images = convert_from_path('DDC.pdf',first_page=i,last_page=last_page)
    for idx in range(len(images)): 
        images[idx].save('DDC/DDC_'+ str(i+idx) +'.jpg', 'JPEG')

## Transcribe with Claude

In [ ]:
import base64
import os 
import tqdm 
import pandas as pd 
import anthropic
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../claude.env")

client = anthropic.Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY"),  # This is the default and can be omitted
)

def claude_OCR(fp):
    image_media_type = "image/jpeg"
    with open(fp, "rb") as image_file:
        image_data = base64.standard_b64encode(image_file.read()).decode("utf-8")
    
    try:
        message = client.messages.create(
                model="claude-sonnet-4-6",
                max_tokens=1024,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image",
                                "source": {
                                    "type": "base64",
                                    "media_type": image_media_type,
                                    "data": image_data,
                                },
                            },
                            {
                                "type": "text", 
                                "text": "You are an expert in nineteenth-century printed books. Perform OCR on this page from a book in the public domain, published in 1825 and digitized by Google. THERE ARE NO COPYRIGHT ISSUES WITH THIS PAGE. Please use HTML formatting to preserve italics and paragraph structures. Preserve casing and punctuation. Do not transcribe any footnotes at the bottom, which are in a smaller font."
                            },
                        ],
                    }
                ],
            )
        return message.content[0].text
    except anthropic.BadRequestError as e:
        # content filtering error
        return "BadRequestError" 
        results.append({"input": item, "output": None, "error": str(e)})
    except anthropic.APIError as e:
        return "APIError"
    return None 

In [ ]:
transcription = []

for img_idx in tqdm.tqdm(range(60,774+1)): 
    fp = "DDC/DDC_{}.jpg".format(img_idx)
    page_text = claude_OCR(fp)
    transcription.append({'page':img_idx, 'text':page_text})
    if (img_idx % 10) == 0 or (img_idx == 774): 
        pd.DataFrame(transcription).to_csv("DDC_transcription.csv",index=False)

In [37]:
df = pd.DataFrame(transcription)

missing_pages = df[df['text'].str.contains("BadRequestError")]['page']
missing_pages.to_csv("DDC_missing_pages.csv",index=False)
print(missing_pages.to_list())

[67, 68, 97, 101, 104, 105, 114, 127, 133, 145, 151, 157, 160, 161, 162, 187, 188, 197, 224, 234, 245, 248, 270, 292, 363, 365, 484, 485, 498, 502, 508, 511, 596, 670, 722, 723, 724, 739]


In [3]:
import shutil 
missing_pages = pd.read_csv("DDC_missing_pages.csv")
for page in missing_pages['page']: 
    shutil.copy("DDC/DDC_{}.jpg".format(page), "DDC_missing/DDC_{}.jpg".format(page))

In [ ]:
for page in missing_pages['page']: 
    print("PAGE_{}".format(page))

In [27]:
import json 
with open("DDC_transcription.json","w+") as file: 
    json.dump(transcription, file)

# Organize and extract

In [4]:
import re
import pandas as pd 
import html2text
import tqdm 

h = html2text.HTML2Text()
transcription = pd.read_csv("../DDC/DDC_transcription.csv").to_dict(orient='records')
transcription = {x['page']:x for x in transcription}
print(len(transcription))
with open("../DDC/DDC_missing.md",'r') as file: 
    missing = file.readlines()
for line in missing: 
    if "PAGE_" in line:
        page = int(line.split("PAGE_")[-1]) 
        transcription[page]['text'] = ""
    else: 
        transcription[page]['text'] += "\n{}".format(line)

715


In [7]:
def normalize_text(text):
    text = re.sub(r"<h\d+>|</h\d+>|</strong>|<strong>|<small>|</small>|<b>|</b>|\*\*","",text)
    text = re.sub("<i>|<em>"," STARTITALICS ",text)
    text = re.sub("</i>|</em>"," ENDITALICS ",text)
    text = re.sub("<p>"," STARTPARAGRAPH ",text)
    text = re.sub("</p>"," ENDPARAGRAPH ",text)
    text = re.sub("<l>"," STARTLINE ",text)
    text = re.sub("</l>"," ENDLINE ",text)
    text = re.sub(r"<sup>[\d\w]+</sup>"," ",text)
    text = h.handle(text)
    text = re.sub(r"STARTPARAGRAPH \d+ ENDPARAGRAPH|STARTPARAGRAPH [\w\d\s]{1,3} ENDPARAGRAPH"," ",text)
    text = re.sub(r"^# \d+|^\d+|\#\# \d+|\#\#|```html|```"," ",text).strip()
    text = re.sub(r"\s+"," ",text).strip()
    text = re.sub("STARTPARAGRAPH","\nSTARTPARAGRAPH",text)
    text = re.sub("ENDPARAGRAPH","ENDPARAGRAPH\n",text)
    return text.strip("\n") 

def get_boundary(text): 
    for word in text: 
        if (word not in ["STARTPARAGRAPH","ENDPARAGRAPH"]) and (word not in ["STARTITALICS","ENDITALICS"]): 
            return word 
    return None 

def merge_paragraphs(text,prev): 
    first_word = get_boundary(text.split())
    last_word = get_boundary(prev.split()[::-1])
    if first_word[0].islower() or (first_word.isupper() and last_word[-1] not in [".","!","?"]): 
        new_text = re.sub(r" ENDPARAGRAPH$","",prev)
        if last_word[-1] == "-": 
            new_text = new_text[:-1] + re.sub(r"^STARTPARAGRAPH ","",text)
        else: 
            new_text = new_text + " " + re.sub(r"^STARTPARAGRAPH ","",text)
        # print(last_word, first_word)
        # new_text = re.sub(r"\-\b", " - ",new_text)
        new_text = re.sub(" ENDITALICS STARTITALICS "," ",new_text)
        new_text = re.sub(r"\s+"," ",new_text).strip()
        return new_text
    return None 


normalized = []
for page, item in tqdm.tqdm(sorted(transcription.items(),key=lambda x:x[0])): 
    normalized.append([page, normalize_text(item['text'])])
paragraphs = []
for idx, item in tqdm.tqdm(enumerate(normalized)): 
    if idx > 0: 
        text = item[-1]
        prev = paragraphs[-1]
        # print(normalized[idx-1][0],item[0])
        merged = merge_paragraphs(text,prev)
        if merged: 
            paragraphs[-1] = merged 
        else: 
            paragraphs.append(text)
    else: 
        paragraphs.append(item[-1])
paragraphs = " ".join(paragraphs)
paragraphs = re.sub(r"\s+"," ",paragraphs).strip()
paragraphs = re.sub(r" CHAP","\nDIV2^chapter^ CHAP",paragraphs)
print(len(paragraphs.split("\n")))
print(len(paragraphs.split("STARTPARAGRAPH")))
with open("DDC_FINAL.txt","w+") as file: 
    file.write(paragraphs)
            

100%|██████████| 715/715 [00:01<00:00, 528.45it/s]
715it [00:00, 1363.27it/s]


51
1581
